<a href="https://colab.research.google.com/github/valerian-drmt/Trading_Projects/blob/main/option_strategy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import

In [ ]:
!pip install QuantLib
!pip install backtrader
!pip install plotly

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import backtrader as bt
import QuantLib as ql
import yfinance as yf

# Fetch Data

In [ ]:
ticker_fx = "EURUSD=X"
ticker_rate = "^IRX"
start_date = "2018-01-01"
end_date = "2025-06-13"

In [ ]:
!pip install pandasdmx --upgrade
!pip install sdmx1 pandas



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.6/244.6 kB 6.2 MB/s eta 0:00:00


In [ ]:
import sdmx
import pandas as pd
import plotly.graph_objects as go

# Connexion à la BCE
client = sdmx.Client("ECB")

# Paramètres de la requête
keys = {
    'FREQ': 'B',
    'REF_AREA': 'U2',
    'CURRENCY': 'EUR',
    'PROVIDER_FM': '4F',
    'INSTRUMENT_FM': 'G_N_A',
    'DATA_TYPE_FM': ['SR_3M', 'SR_6M', 'SR_1Y']
}

response = client.data(
    resource_id='YC',
    key=keys,
    params={'startPeriod': '2020-01-01'}
)

# Conversion en DataFrame
df = sdmx.to_pandas(response, datetime={"dim": "TIME_PERIOD"})
# Aplatir les colonnes MultiIndex
df.columns = df.columns.get_level_values('DATA_TYPE_FM')

# Renommer les colonnes pour plus de clarté
df = df.rename(columns={
    'SR_3M': 'EUR_3M',
    'SR_6M': 'EUR_6M',
    'SR_1Y': 'EUR_1Y'
})

# Vérifie le résultat
print(df.head())


# Plotly chart
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['EUR_3M'], mode='lines', name='EUR Rate 3M'))
fig.add_trace(go.Scatter(x=df.index, y=df['EUR_6M'], mode='lines', name='EUR Rate 6M'))
fig.add_trace(go.Scatter(x=df.index, y=df['EUR_1Y'], mode='lines', name='EUR Rate 1Y'))

fig.update_layout(
    title='EUR OIS Rates (3M, 6M, 1Y) from ECB',
    xaxis_title='Date',
    yaxis_title='Rate (%)',
    legend_title='Maturity',
    template='plotly_white'
)

fig.show()


DATA_TYPE_FM    EUR_1Y    EUR_3M    EUR_6M
TIME_PERIOD                               
2020-01-02   -0.634844 -0.623324 -0.629533
2020-01-03   -0.630324 -0.595449 -0.610370
2020-01-06   -0.637580 -0.597296 -0.614162
2020-01-07   -0.641443 -0.605300 -0.620692
2020-01-08   -0.633612 -0.597340 -0.612942


In [ ]:
import pandas as pd
from pandas_datareader import data as web
from datetime import datetime

# Période souhaitée
start = datetime(2020, 1, 1)
end = datetime.today()

# Codes FRED pour les taux souverains US
tickers = {
    '3M': 'DGS3MO',
    '6M': 'DGS6MO',
    '1Y': 'DGS1',
}

# Récupération des données
df = web.DataReader(list(tickers.values()), 'fred', start, end)

# Renommage des colonnes
df.columns = tickers.keys()

# Suppression des lignes vides
df = df.dropna(how='all')

print(df.tail())



fig = go.Figure()
# Add USD Rate relative performance line
fig.add_trace(go.Scatter(x=df.index, y=df['3M'], mode='lines', name='USD Rate 3M'))
fig.add_trace(go.Scatter(x=df.index, y=df['6M'], mode='lines', name='USD Rate 3M'))
fig.add_trace(go.Scatter(x=df.index, y=df['1Y'], mode='lines', name='USD Rate 3M'))
# Customize layout
fig.update_layout(
    title='USD 3M Rate Over Time',
    xaxis_title='Date',
    yaxis_title='Rate',
    legend_title='Legend',
    template='plotly'
)
# Show the plot
fig.show()

              3M    6M    1Y
DATE                        
2025-06-10  4.45  4.32  4.12
2025-06-11  4.45  4.31  4.08
2025-06-12  4.46  4.29  4.06
2025-06-13  4.45  4.30  4.09
2025-06-16  4.43  4.32  4.10


In [ ]:
# Download data
eurusd = yf.download(ticker_fx, start=start_date, end=end_date)

# Data cleaning and preparation
eurusd_close = eurusd['Close'].dropna()

data = pd.DataFrame()
data['EURUSD'] = eurusd_close
data.dropna(inplace=True)
print("\n")
print(data.shape)
print(data.head())

# Create the plot
fig = go.Figure()
# Add EUR/USD relative performance line
fig.add_trace(go.Scatter(x=data.index, y=data['EURUSD'], mode='lines', name='EUR/USD'))

# Customize layout
fig.update_layout(
    title='EUR/USD',
    xaxis_title='Date',
    yaxis_title='Price',
    legend_title='Legend',
    template='plotly'
)
# Show the plot
fig.show()

/tmp/ipython-input-46-3848379024.py:2: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed



(1940, 1)
              EURUSD
Date                
2018-01-01  1.200495
2018-01-02  1.201158
2018-01-03  1.206345
2018-01-04  1.201043
2018-01-05  1.206884


# Pricing Options

## European Simple Option

In [ ]:
def european_fx_option_pricing(option_type, S, K, r_usd, r_eur, T, sigma, today):
    """
    Price a European FX option (e.g., EUR/USD) using Black-Scholes-Merton model.

    Parameters:
    - option_type: 'call' or 'put' (call on EUR/USD = right to buy EUR, sell USD)
    - S: Spot exchange rate (e.g., EUR/USD spot rate in USD per EUR)
    - K: Strike exchange rate
    - r_usd: USD interest rate (domestic rate for EUR/USD)
    - r_eur: EUR interest rate (foreign rate for EUR/USD)
    - T: Time to maturity in years
    - sigma: Volatility of the exchange rate
    - today: QuantLib Date object for evaluation date

    Returns:
    - Tuple of (option_price, delta, gamma, theta, vega)
    """
    # Input validation
    if S <= 0 or K <= 0:
        raise ValueError("Spot price (S) and strike price (K) must be positive.")
    if sigma <= 0:
        raise ValueError("Volatility (sigma) must be positive.")
    if T <= 0:
        raise ValueError("Time to maturity (T) must be positive.")

    # 1. Set the evaluation date
    ql.Settings.instance().evaluationDate = today

    # 2. Define the option type
    if option_type.lower() == 'call':
        option_type_ql = ql.Option.Call  # Call on EUR/USD: right to buy EUR
    elif option_type.lower() == 'put':
        option_type_ql = ql.Option.Put  # Put on EUR/USD: right to sell EUR
    else:
        raise ValueError("Invalid option type. Must be 'call' or 'put'.")

    # 3. Create the option object
    payoff = ql.PlainVanillaPayoff(option_type_ql, K)
    # Calculate maturity date (use calendar for accuracy)
    exercise_date = today + ql.Period(int(T * 360), ql.Days)  # Approximate T in days
    exercise = ql.EuropeanExercise(exercise_date)
    european_option = ql.VanillaOption(payoff, exercise)

    # 4. Define the market data
    spot_handle = ql.QuoteHandle(ql.SimpleQuote(S))
    # Domestic rate (USD for EUR/USD)
    rate_handle = ql.YieldTermStructureHandle(ql.FlatForward(today, r_usd, ql.Actual365Fixed()))
    # Foreign rate (EUR for EUR/USD)
    dividend_handle = ql.YieldTermStructureHandle(ql.FlatForward(today, r_eur, ql.Actual365Fixed()))

    vol_handle = ql.BlackVolTermStructureHandle(ql.BlackConstantVol(today, ql.NullCalendar(), sigma, ql.Actual365Fixed()))

    # 5. Create the Black-Scholes process for FX
    bsm_process = ql.BlackScholesMertonProcess(spot_handle, dividend_handle, rate_handle, vol_handle)

    # 6. Create the pricing engine
    engine = ql.AnalyticEuropeanEngine(bsm_process)

    # 7. Set the pricing engine for the option
    european_option.setPricingEngine(engine)

    # 8. Calculate the option price and Greeks
    option_price = european_option.NPV()  # Price in USD per EUR
    delta = european_option.delta()  # Delta w.r.t. spot exchange rate
    gamma = european_option.gamma()  # Gamma w.r.t. spot exchange rate
    theta = european_option.theta() / 365  # Annualized theta (USD per EUR per year)
    vega = european_option.vega() / 100   # Vega per 1% change in volatility

    return option_price, delta, gamma, theta, vega


In [ ]:
# Initialize DataFrame to store straddle price and Greeks
Option_Price_List = pd.DataFrame(
    columns=["Straddle_3M_ATM", "Delta", "Gamma", "Theta", "Vega"],
    index=data.index
)

vol = data["EURUSD"].pct_change().dropna().std() * np.sqrt(252)  # Annualized volatility
print(f"Calculated volatility: {vol}")
T = 3/12  # 3 months
q = 0.03  # No dividend yield

# Iterate over DataFrame
for index, row in data.iterrows():
    S = row["EURUSD"]
    K = row["EURUSD"]*np.exp(-row["USD_Rate_3M"]*T)
    r = row["USD_Rate_3M"]
    sigma = vol
    today = ql.Date(index.day, index.month, index.year)

    try:
        # Price call and put options
        option_price_call_atm, delta_call, gamma_call, theta_call, vega_call = european_fx_option_pricing(
            "call", S, K, r, q, T, sigma, today
        )
        option_price_put_atm, delta_put, gamma_put, theta_put, vega_put = european_fx_option_pricing(
            "put", S, K, r, q, T, sigma, today
        )
        # Calculate straddle price and Greeks
        straddle_price = option_price_call_atm + option_price_put_atm
        straddle_delta = delta_call + delta_put
        straddle_gamma = gamma_call + gamma_put
        straddle_theta = theta_call + theta_put
        straddle_vega = vega_call + vega_put

        # Store in DataFrame
        Option_Price_List.loc[index] = [
            straddle_price,
            straddle_delta,
            straddle_gamma,
            straddle_theta,
            straddle_vega
        ]
    except Exception as e:
        print(f"Error pricing for date {index}: {e}")
        Option_Price_List.loc[index] = [None, None, None, None, None]

Option_Price_List["EURUSD"] = data["EURUSD"]
Option_Price_List["USD_Rate_3M"] = data["USD_Rate_3M"]
print(Option_Price_List.head())

# Créer le graphique avec Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=Option_Price_List.index, y=Option_Price_List["EURUSD"] + Option_Price_List["Straddle_3M_ATM"], mode='lines', name='Upper Bound'))
fig.add_trace(go.Scatter(x=Option_Price_List.index, y=Option_Price_List["EURUSD"], mode='lines', name='EURUSD'))
fig.add_trace(go.Scatter(x=Option_Price_List.index, y=Option_Price_List["EURUSD"] - Option_Price_List["Straddle_3M_ATM"], mode='lines', name='Lower Bound'))

fig.update_layout(
    title="Option Price Over Time",
    xaxis_title="Date",
    yaxis_title="Option Price"
)

fig.show()

Calculated volatility: 0.07388815445845266
           Straddle_3M_ATM     Delta      Gamma     Theta      Vega    EURUSD  \
Date                                                                            
2018-01-02        0.034916  0.002561  17.971133 -0.000192  0.004724  1.201158   
2018-01-03        0.035068  0.001703  17.893906 -0.000193  0.004744  1.206345   
2018-01-04        0.034914  0.001703  17.972912 -0.000192  0.004723  1.201043   
2018-01-05        0.035084  0.001703   17.88592 -0.000193  0.004746  1.206884   
2018-01-08        0.034991  0.002775  17.932477 -0.000193  0.004734  1.203746   

            USD_Rate_3M  
Date                     
2018-01-02      0.01378  
2018-01-03      0.01370  
2018-01-04      0.01370  
2018-01-05      0.01370  
2018-01-08      0.01380  


## Pivot Point Indicateur